# Tensor analysis of Argo

This notebook illustrates how a tensor modeling approach can be used to understand the roles of time and space in the Argo temperature or salinity data.  We use an approach called [parafac](https://en.wikipedia.org/wiki/Tensor_rank_decomposition), which approximates a data tensor as a low rank tensor, analogous to what the SVD does for matrices.

We construct an order 4 tensor D such that D[i, j, k, l] contains the Mahalanobis distance between profiles in stratum (i, j) and stratum (k, l).  In the index tuple (i, j), i corresponds to the latitude stratum and j corresponds to the day of year stratum (day of year counts the days in the year from 1 to 365).  The four indices i, j, k, l are often referred to as "modes" in this type of analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from read import *
from tensorly.decomposition import parafac

In [ ]:
def mahalanobis_distance(X, Y, ratio=1e-3):
    """
    Compute the squared Mahalanobis distance between the observations in X and Y.

    The observations are in the columns of X and Y.  The covariance estimate 
    is regularized so that eigenvalues smaller than 'ratio' times the leading
    eigenvalue of the pooled covariance matrix are set to zero.  Then the 
    pseudo-inverse is used for estimating the Mahalanobis distance.  The 
    result is equivalent to projecting the data onto the eigenspace spanned 
    by eigenvectors whose eigenvalue is greater than 'ratio' times the leading 
    eigenvalue of the pooled covariance matrix.
    """
    mux = X.mean(1)
    muy = Y.mean(1)
    X = X - mux[:, None]
    Y = Y - muy[:, None]
    S = np.dot(X, X.T) + np.dot(Y, Y.T)
    S /= (X.shape[1] + Y.shape[1])
    a, b = np.linalg.eigh(S)
    ii = np.flatnonzero(a >= a[-1]*ratio)
    a = a[ii]
    b = b[:, ii]
    Sinv = np.dot(b, (1 / a[:, None]) * b.T)
    dmu = mux - muy
    return np.dot(dmu, np.dot(Sinv, dmu))

In [ ]:
def make_tensor(X, b=10):
    """
    Create an order 4 tensor containing the Mahalanobis distances between pairs
    of strata defined by latitude and day of year.  Use 'b' bins along both
    axes.  Returns tensors containing the Mahalanobis distances and the sample
    sizes used for estimation.

    D[i, j, k, l] is the Mahalanobis distance between stratum (i, j) and stratum
    (k, l).  The two indices defining a stratum correspond to latitude and day
    respectively.
    """
    dd = pd.DataFrame({"id": np.arange(len(lat)), "lat": lat, "day": day})
    dd["lat_bin"] = pd.qcut(dd["lat"], q=b)
    dd["day_bin"] = pd.qcut(dd["day"], q=b)

    lat_bins = dd["lat_bin"].unique().tolist()
    lat_bins.sort()
    day_bins = dd["day_bin"].unique().tolist()
    day_bins.sort()
    gr = dd.groupby(["lat_bin", "day_bin"]).groups

    grk = [x for x in gr.keys()]
    D = np.zeros((b, b, b, b))
    N1 = np.zeros((b, b, b, b))
    N2 = np.zeros((b, b, b, b))    
    for (i,ki) in enumerate(grk):
        i0 = lat_bins.index(grk[i][0])
        i1 = day_bins.index(grk[i][1])
        for j in range(i):
            kj = grk[j]
            i2 = lat_bins.index(grk[j][0])
            i3 = day_bins.index(grk[j][1])
            d = mahalanobis_distance(X[:, gr[ki]], X[:, gr[kj]])
            D[i0, i1, i2, i3] = d
            D[i2, i3, i0, i1] = d
            N1[i0, i1, i2, i3] = len(gr[ki])
            N1[i2, i3, i0, i1] = len(gr[ki])
            N2[i0, i1, i2, i3] = len(gr[kj])
            N2[i2, i3, i0, i1] = len(gr[kj])
    return D, N1, N2

Construct the tensor for the temperature data, and center it along each axis.  The parafac decomposition of a centered distance matrix is essentially a tensor multidimensional scaling.

In [ ]:
D, N1, N2 = make_tensor(temp)
B = D - D.mean()
for k in range(len(B.shape)):
    B -= B.mean(axis=k, keepdims=True)

Perform the parafac analysis to approximate B as a tensor of specified rank.  Standardize the factors to be unit vectors and factor out the overall scale as 'lam'.

In [ ]:
def parafacx(B, rank=1, fixed_modes=[]):
    if len(fixed_modes) == 0:
        weights, factors = parafac(B, rank=rank)
    else:
        weights, factors = parafac(B, rank=rank, fixed_modes=fixed_modes)

    lam = np.ones(rank)
    for j in range(len(factors)):
        for k in range(rank):
            v = np.linalg.norm(factors[j][:, k])
            factors[j][:, k] /= v
            lam[k] *= v

    return lam, factors

These functions can be used to reconstruct a tensor from a parafac factorization.

In [ ]:
def predict_term(lam, factors, j):
    from functools import reduce
    return lam[j] * reduce(np.multiply.outer, [f[:, j] for f in factors]) 

def predict(lam, factors):
    tl = [predict_term(lam, factors, j) for j in range(len(lam))]
    return np.sum(tl, axis=0)

Below we can see how the fit is improved as higher order approximations are used.

In [ ]:
e = np.sqrt((B**2).sum())
print(f"{e:.2f}")
for r in [1, 2, 3]:
    lam, factors = parafacx(B, rank=r)
    Bhat = predict(lam, factors)
    e = np.sqrt(((B-Bhat)**2).sum())
    print(f"{e:.2f}")

The distance tensor D has a symmetry property in that D[i, j, k, l] = D[k, l, i, j].  As a result, the loadings for the first and third modes should be equal up to sign and the loadings for the second and fourth modes should be equal up to sign.  This is only guaranteed to hold for the rank one parafac approximation.

In [ ]:
lam, factors = parafacx(B, rank=1)
Q = np.zeros((4, 4))
for j in range(4):
    for k in range(4):
        Q[j, k] = min(np.linalg.norm(factors[j] - factors[k]), 
                      np.linalg.norm(factors[j] + factors[k]))
Q

The function below standardizes a rank one parafac fit so that the loaings for the first and third modes are equal and the loadings for the second and fourth modes are equal.

In [ ]:
def standardize_rank1(lam, factors):
    assert(len(lam) == 1)
    for j in [0,1]:
        q1 = np.linalg.norm(factors[j] - factors[j+2])
        q2 = np.linalg.norm(factors[j] + factors[j+2])
        assert(min(abs(q1), abs(q2)) < 1e-2)
        if q2 < q1:
            factors[j+2] *= -1
            lam[0] *= -1
    return lam, factors

After standardization, the value of lambda is negative, which is important for reasons that will be discussed below.

In [ ]:
lam, factors = standardize_rank1(lam, factors)
lam

These are what the loadings look like for the latitude factor and the day of year factor.  Because of the sinusoidal pattern for the day of year effect, the approximated squared Mahalanobis distances will be maximized wben comparing winter (stratum 1) to summer (strata 6-7).  The loading values at these points have maximal magnitudes and opposite signs.  Thus their product will have maximal magnitude and a negative sign, and finally the fitted value becomes positive when multiplying by 'lam'.

In [ ]:
plt.grid(True)
for j,f in enumerate(factors[0:2]):
    plt.plot(f, label=["Lat", "Day"][j])
ha, lb = plt.gca().get_legend_handles_labels()
plt.figlegend(ha, lb)
plt.xlabel("Band")
plt.ylabel("Loading")

We can also compare the goodness of fit for constrained models, in which some of the modes are held constant.  We can see that holding day of year fixed has a lower cost than holding latitude fixed, showing that day of year contributes less to the fit than latitude.

In [ ]:
# Swap day of year with latitude.
B1 = np.swapaxes(B, axis1=0, axis2=1)
B1 = np.swapaxes(B1, axis1=2, axis2=3)

for j,Bx in enumerate([B, B1]):
    lam1, factors1 = parafacx(Bx, rank=1, fixed_modes=[0,2])
    Bhat1 = predict(lam1, factors1)
    print("RMSE holding %s fixed:" % ("latitude" if j == 0 else "day of year"))
    print(np.sqrt(np.sum((Bx - Bhat1)**2)))